In [1]:
import numpy as np
import networkx as nx
from Modules.brute_force_bond_finder import brute_force_bond_finder, bond_sorter
from Modules.meet_sublattice_machinery import partition_meet, bond_meet

In [2]:
def partition_vi(p_1, p_2):

    """
        Calculates the partition variation of information given two partitions (clusterings) p_1 and p_2.
    """

    # Get number of vertices in graph
    n = len([x for block in p_1 for x in block])

    # Calculate p_1 and p_2 discrete distribution
    p_1_dist = np.asarray([len(block) / n for block in p_1])
    p_2_dist = np.asarray([len(block) / n for block in p_2])

    # Calculate partition meet distribution
    meet = partition_meet(p_1, p_2)
    meet_dist = np.asarray([len(block) / n for block in meet])

    # Entropies
    H_p_1 = - np.sum(p_1_dist * np.log(p_1_dist))
    H_p_2 = - np.sum(p_2_dist * np.log(p_2_dist))
    H_meet = - np.sum(meet_dist * np.log(meet_dist))

    return 2 * H_meet - H_p_1 - H_p_2

In [3]:
def bond_vi(p_1, p_2, G, algo):

    """
        Calculates the partition variation of information given two partitions (clusterings) p_1 and p_2.
    """

    # Get number of vertices in graph
    n = len([x for block in p_1 for x in block])

    # Calculate p_1 and p_2 discrete distribution
    p_1_dist = np.asarray([len(block) / n for block in p_1])
    p_2_dist = np.asarray([len(block) / n for block in p_2])

    # Calculate partition meet distribution
    meet = bond_meet(p_1, p_2, G, algo)
    meet_dist = np.asarray([len(block) / n for block in meet])

    # Entropies
    H_p_1 = - np.sum(p_1_dist * np.log(p_1_dist))
    H_p_2 = - np.sum(p_2_dist * np.log(p_2_dist))
    H_meet = - np.sum(meet_dist * np.log(meet_dist))

    return 2 * H_meet - H_p_1 - H_p_2

In [4]:
def VI_equality_checker(G, algo):

    """
        Checks whether the partition variation of information is equal to the bond variation of
        information for a given graph family, up to number of vertices d
    """
    
    bonds = brute_force_bond_finder(G)
    part_vis = []
    bond_vis = []
    for bond_1 in bonds:    
        for bond_2 in bonds:
            p_vi = partition_vi(bond_1, bond_2)
            b_vi = bond_vi(bond_1, bond_2, G, algo)
            
            part_vis.append(p_vi)
            bond_vis.append(b_vi)

    return np.isclose(bond_vis, part_vis).all()

In [46]:
checks = []
for i in range(7):
    graph = nx.complete_graph(i)
    check = VI_equality_checker(graph, brute_force_bond_finder)
    checks.append(check)
print(checks)

[np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_]


In [47]:
checks = []
for i in range(7):
    graph = nx.path_graph(i)
    check = VI_equality_checker(graph, brute_force_bond_finder)
    checks.append(check)
print(checks)

[np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_]


In [49]:
checks = []
for i in range(7):
    graph = nx.star_graph(i)
    check = VI_equality_checker(graph, brute_force_bond_finder)
    checks.append(check)
print(checks)

[np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_]


In [50]:
checks = []
for i in range(7):
    graph = nx.wheel_graph(i)
    check = VI_equality_checker(graph, brute_force_bond_finder)
    checks.append(check)
print(checks)

[np.True_, np.True_, np.True_, np.True_, np.True_, np.False_, np.False_]


In [51]:
checks = []
for i in range(7):
    graph = nx.cycle_graph(i)
    check = VI_equality_checker(graph, brute_force_bond_finder)
    checks.append(check)
print(checks)

[np.True_, np.True_, np.True_, np.True_, np.False_, np.False_, np.False_]


In [13]:
C_4 = nx.cycle_graph(4)
bonds = brute_force_bond_finder(C_4)
violations = []
for b1 in bonds:
    for b2 in bonds:
        for b3 in bonds:
            d12 = bond_vi(b1, b2, C_4, brute_force_bond_finder)
            d13 = bond_vi(b1, b3, C_4, brute_force_bond_finder)
            d23 = bond_vi(b2, b3, C_4, brute_force_bond_finder)
            if d12 > d13 + d23 + 1e-9:
                violations.append((b1, b2, b3, d12, d13, d23, d12 - (d13 + d23)))

for v in violations:
    print(v)

(frozenset({frozenset({1, 2, 3}), frozenset({0})}), frozenset({frozenset({2}), frozenset({0, 1, 3})}), frozenset({frozenset({0, 1, 2, 3})}), np.float64(1.6479184330021648), np.float64(0.5623351446188083), np.float64(0.5623351446188083), np.float64(0.5232481437645482))
(frozenset({frozenset({1}), frozenset({0, 2, 3})}), frozenset({frozenset({3}), frozenset({0, 1, 2})}), frozenset({frozenset({0, 1, 2, 3})}), np.float64(1.6479184330021648), np.float64(0.5623351446188083), np.float64(0.5623351446188083), np.float64(0.5232481437645482))
(frozenset({frozenset({3}), frozenset({0, 1, 2})}), frozenset({frozenset({1}), frozenset({0, 2, 3})}), frozenset({frozenset({0, 1, 2, 3})}), np.float64(1.6479184330021648), np.float64(0.5623351446188083), np.float64(0.5623351446188083), np.float64(0.5232481437645482))
(frozenset({frozenset({2}), frozenset({0, 1, 3})}), frozenset({frozenset({1, 2, 3}), frozenset({0})}), frozenset({frozenset({0, 1, 2, 3})}), np.float64(1.6479184330021648), np.float64(0.5623351